In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)
postgres_url = os.getenv("POSTGRES_URL")

In [3]:
postgres_engine = create_engine(postgres_url, pool_pre_ping=True)

In [4]:
rfm_query = """
WITH snapshot AS (
    -- Reference anchor date: 1 day past the dataset's latest recorded purchase
    SELECT MAX(order_purchase_timestamp::timestamp)::date + INTERVAL '1 day' AS snapshot_date
    FROM olist_ecommerce.orders
),
customer_rfm_raw AS (
    -- Aggregate raw Recency, Frequency, and Monetary metrics per unique customer
    SELECT 
        c.customer_unique_id,
        (SELECT snapshot_date FROM snapshot) - MAX(o.order_purchase_timestamp::timestamp)::date AS recency_days,
        COUNT(DISTINCT o.order_id) AS frequency,
        ROUND(SUM(p.payment_value)::numeric, 2) AS monetary
    FROM olist_ecommerce.orders o
    JOIN olist_ecommerce.customers c ON o.customer_id = c.customer_id
    JOIN olist_ecommerce.order_payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
),
rfm_scores AS (
    -- Calculate 1 to 5 quantile scores using NTILE(5)
    SELECT 
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM customer_rfm_raw
)
-- Combine individual scores into an RFM segment string
SELECT 
    customer_unique_id,
    recency_days,
    frequency,
    monetary,
    r_score,
    f_score,
    m_score,
    CONCAT(r_score, f_score, m_score) AS rfm_segment
FROM rfm_scores
ORDER BY monetary DESC
LIMIT 15;
"""

In [5]:
rfm_df = pd.read_sql(rfm_query, con=postgres_engine)

In [6]:
rfm_df

,customer_unique_id,recency_days,frequency,monetary,r_score,f_score,m_score,rfm_segment
0,0a0a92112bd4c708ca5fde585afaa872,384 days,1,13664.08,2,5,5,255
1,da122df9eeddfedc1dc1f5349a1a690c,565 days,2,7571.63,1,5,5,155
2,763c8b1c9c68a0229c42c9fc6f662b93,95 days,1,7274.88,5,5,5,555
3,dc4802a71eae9be1dd28f5d788ceb526,613 days,1,6929.31,1,5,5,155
4,459bef486812aa25204be022145caa62,85 days,1,6922.21,5,5,5,555
5,ff4159b92c40ebe40454e3e6a7c35ed6,512 days,1,6726.66,1,5,5,155
6,4007669dec559734d6f53e029e360987,328 days,1,6081.54,2,5,5,255
7,eebb5dda148d3893cdaf5b5ca3040ccb,548 days,1,4764.34,1,5,5,155
8,48e1ac109decbb87765a3eade6854098,118 days,1,4681.78,5,5,5,555
9,c8460e4251689ba205045f3ea17884a1,71 days,4,4655.91,5,5,5,555


In [9]:
from db_utils import run_pg_query

In [11]:
df = run_pg_query(rfm_query)
df

,customer_unique_id,recency_days,frequency,monetary,r_score,f_score,m_score,rfm_segment
0,0a0a92112bd4c708ca5fde585afaa872,384 days,1,13664.08,2,5,5,255
1,da122df9eeddfedc1dc1f5349a1a690c,565 days,2,7571.63,1,5,5,155
2,763c8b1c9c68a0229c42c9fc6f662b93,95 days,1,7274.88,5,5,5,555
3,dc4802a71eae9be1dd28f5d788ceb526,613 days,1,6929.31,1,5,5,155
4,459bef486812aa25204be022145caa62,85 days,1,6922.21,5,5,5,555
5,ff4159b92c40ebe40454e3e6a7c35ed6,512 days,1,6726.66,1,5,5,155
6,4007669dec559734d6f53e029e360987,328 days,1,6081.54,2,5,5,255
7,eebb5dda148d3893cdaf5b5ca3040ccb,548 days,1,4764.34,1,5,5,155
8,48e1ac109decbb87765a3eade6854098,118 days,1,4681.78,5,5,5,555
9,c8460e4251689ba205045f3ea17884a1,71 days,4,4655.91,5,5,5,555
